##  製作數據庫


In [1]:
import cv2
import numpy as np
import os
import pandas as pd

def calculate_frame_difference(prev_frame, current_frame):
    diff = cv2.absdiff(prev_frame, current_frame)
    epsilon = 1e-10  # 小值防止除以零
    return np.sum(diff / ( prev_frame + epsilon))

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    differences = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = calculate_frame_difference(prev_frame, current_frame)
        differences.append(diff)
        prev_frame = current_frame

    cap.release()
    return differences

def calculate_thresholds(differences):
    high_threshold = np.percentile(differences, 70)
    low_threshold = np.percentile(differences, 30)
    return high_threshold, low_threshold

def classify_activities(differences, low_threshold, high_threshold):
    activities = []
    for diff in differences:
        if diff >= high_threshold:
            activities.append(('Aggressive', diff))
        elif diff >= low_threshold:
            activities.append(('Normal', diff))
        else:
            activities.append(('Passive', diff))
    return activities

def process_folder(folder_path):
    all_differences = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            all_differences.extend(differences)

    high_threshold, low_threshold = calculate_thresholds(all_differences)
    all_results = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            activities = classify_activities(differences, low_threshold, high_threshold)
            results = [{'video': os.path.basename(video_path), 'activity': act[0], 'difference': act[1]} for act in activities]
            all_results.extend(results)

    return all_results

def save_results_to_excel(results, output_file):
    df = pd.DataFrame(results)
    df.to_excel(output_file, index=False)

# 使用范例
folder_path = r'C:\Users\skybl\Desktop\進階專題\New_影片資料夾'
results = process_folder(folder_path)
output_file = 'output_results.xlsx'
save_results_to_excel(results, output_file)


## 個別分析影片

In [2]:
import pandas as pd

data = pd.read_excel(r'C:\Users\skybl\Desktop\進階專題\程式碼區\output_results.xlsx')
print(data.columns)


Index(['video', 'activity', 'difference'], dtype='object')


In [3]:
grouped = data.groupby('video')
dataframes = {name: group for name, group in grouped}
dataframes

{'亂走亂舔.mp4':         video activity    difference
 0    亂走亂舔.mp4  Passive  2.577000e+13
 1    亂走亂舔.mp4  Passive  1.860000e+12
 2    亂走亂舔.mp4  Passive  2.561000e+13
 3    亂走亂舔.mp4  Passive  2.750000e+12
 4    亂走亂舔.mp4  Passive  2.162000e+13
 ..        ...      ...           ...
 348  亂走亂舔.mp4  Passive  2.800000e+13
 349  亂走亂舔.mp4  Passive  1.400000e+11
 350  亂走亂舔.mp4  Passive  3.305000e+13
 351  亂走亂舔.mp4  Passive  2.140000e+12
 352  亂走亂舔.mp4  Passive  2.605000e+13
 
 [353 rows x 3 columns],
 '互舔.mp4':       video    activity    difference
 353  互舔.mp4     Passive  3.284000e+13
 354  互舔.mp4      Normal  5.316000e+13
 355  互舔.mp4      Normal  6.847000e+13
 356  互舔.mp4  Aggressive  1.122740e+15
 357  互舔.mp4      Normal  1.230800e+14
 ..      ...         ...           ...
 872  互舔.mp4     Passive  2.520000e+13
 873  互舔.mp4      Normal  5.173000e+13
 874  互舔.mp4      Normal  7.107000e+13
 875  互舔.mp4      Normal  6.973000e+13
 876  互舔.mp4     Passive  5.008000e+13
 
 [524 rows x 3 columns],


In [4]:
import pandas as pd

# 加载Excel文件
data = pd.read_excel('./output_results.xlsx')

# 确保数据中的列名是正确的，特别是关注 'video' 和 'activity' 列
print(data.head())
print(data.columns)

# 按视频名分组，并计算每个视频中不同活动状态的数量
grouped = data.groupby('video')
activity_counts = grouped['activity'].value_counts().unstack(fill_value=0)

# 创建 DataFrame 存储 activity_counts
activity_counts_df = pd.DataFrame(activity_counts)
activity_counts_df

      video activity    difference
0  亂走亂舔.mp4  Passive  2.577000e+13
1  亂走亂舔.mp4  Passive  1.860000e+12
2  亂走亂舔.mp4  Passive  2.561000e+13
3  亂走亂舔.mp4  Passive  2.750000e+12
4  亂走亂舔.mp4  Passive  2.162000e+13
Index(['video', 'activity', 'difference'], dtype='object')


activity,Aggressive,Normal,Passive
video,,,
亂走亂舔.mp4,15,84,254
互舔.mp4,29,357,138
互舔＋到處晃晃.mp4,428,487,2
休息.mp4,200,420,293
休息2.mp4,21,109,483
休息吃草.mp4,118,322,33
休息走路.mp4,189,543,197
休息＋人經過轉頭.mp4,25,117,470
伸懶腰.mp4,134,166,1


In [5]:
import pandas as pd

# 加载Excel文件
data = pd.read_excel('./output_results.xlsx')

# 打印数据的前几行和所有列名，确保数据正确加载
print(data.head())
print(data.columns)

# 确保 'video' 和 'activity' 列名正确
if 'video' in data.columns and 'activity' in data.columns:
    # 按视频名分组，并计算每个视频中不同活动状态的数量
    grouped = data.groupby('video')
    activity_counts = grouped['activity'].value_counts().unstack(fill_value=0)

    # 计算每个视频中活动状态的百分比，并四舍五入到整数位
    activity_percentages = (activity_counts.div(activity_counts.sum(axis=1), axis=0) * 100).round(0).astype(int)

    # 创建 DataFrame 存储 activity_percentages
    activity_percentages_df = pd.DataFrame(activity_percentages)

    # 打印结果
    print(activity_percentages_df)
else:
    print("列名 'video' 或 'activity' 不存在，请检查Excel文件中的列名。")


      video activity    difference
0  亂走亂舔.mp4  Passive  2.577000e+13
1  亂走亂舔.mp4  Passive  1.860000e+12
2  亂走亂舔.mp4  Passive  2.561000e+13
3  亂走亂舔.mp4  Passive  2.750000e+12
4  亂走亂舔.mp4  Passive  2.162000e+13
Index(['video', 'activity', 'difference'], dtype='object')
activity        Aggressive  Normal  Passive
video                                      
亂走亂舔.mp4                 4      24       72
互舔.mp4                   6      68       26
互舔＋到處晃晃.mp4             47      53        0
休息.mp4                  22      46       32
休息2.mp4                  3      18       79
休息吃草.mp4                25      68        7
休息走路.mp4                20      58       21
休息＋人經過轉頭.mp4             4      19       77
伸懶腰.mp4                 45      55        0
動來動去.mp4                13      65       22
受水驚嚇.mp4                87      12        0
吃草休息.mp4                36      62        2
喝水＋聞屁.mp4                5      37       58
抬腳抓癢(無裁切版）.mp4          23      75        1
搖尾巴＋吃草.mp4              41 

In [6]:
activity_percentages_df

activity,Aggressive,Normal,Passive
video,,,
亂走亂舔.mp4,4,24,72
互舔.mp4,6,68,26
互舔＋到處晃晃.mp4,47,53,0
休息.mp4,22,46,32
休息2.mp4,3,18,79
休息吃草.mp4,25,68,7
休息走路.mp4,20,58,21
休息＋人經過轉頭.mp4,4,19,77
伸懶腰.mp4,45,55,0


In [7]:
activity_percentages.shape, activity_percentages_df.columns

((24, 3),
 Index(['Aggressive', 'Normal', 'Passive'], dtype='object', name='activity'))

In [8]:
# 计算每个视频中活动状态的百分比，并四舍五入到整数位
activity_percentages = (activity_counts.div(activity_counts.sum(axis=1), axis=0) * 100).round(0).astype(int)

# 将超过 75% 的状态设为主要状态，否则设为 "Normal"
main_activity = activity_percentages.idxmax(axis=1)
main_activity_percentage = activity_percentages.max(axis=1)
main_activity[main_activity_percentage <= 75] = 'Normal'

# 创建新的 DataFrame 记录视频及其主要活动状态
new_data = pd.DataFrame({'video': main_activity.index, 'main_activity': main_activity.values,'Aggressive':activity_percentages['Aggressive'],'Normal':activity_percentages['Normal'],'Passive':activity_percentages['Passive']})



# 打印新的 DataFrame
new_data


,video,main_activity,Aggressive,Normal,Passive
video,,,,,
亂走亂舔.mp4,亂走亂舔.mp4,Normal,4,24,72
互舔.mp4,互舔.mp4,Normal,6,68,26
互舔＋到處晃晃.mp4,互舔＋到處晃晃.mp4,Normal,47,53,0
休息.mp4,休息.mp4,Normal,22,46,32
休息2.mp4,休息2.mp4,Passive,3,18,79
休息吃草.mp4,休息吃草.mp4,Normal,25,68,7
休息走路.mp4,休息走路.mp4,Normal,20,58,21
休息＋人經過轉頭.mp4,休息＋人經過轉頭.mp4,Passive,4,19,77
伸懶腰.mp4,伸懶腰.mp4,Normal,45,55,0


## K-means 聚類

資料處理

In [9]:
import cv2
import numpy as np
import os
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def calculate_frame_difference(prev_frame, current_frame):
    diff = cv2.absdiff(prev_frame, current_frame)
    epsilon = 1e-10  # 小值防止除以零
    return np.sum(diff / ( prev_frame + epsilon))

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    differences = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = calculate_frame_difference(prev_frame, current_frame)
        differences.append(diff)
        prev_frame = current_frame

    cap.release()
    return differences

def process_folder(folder_path):
    all_differences = {}
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            all_differences[filename] = differences

    all_differences_list = [np.mean(diffs) for diffs in all_differences.values()]

    # 特徵縮放
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(np.array(all_differences_list).reshape(-1, 1))

    # 應用 K-means 聚類
    kmeans = KMeans(n_clusters=3, random_state=42)
    kmeans.fit(scaled_features)
    
    # 獲取聚類標籤
    cluster_labels = kmeans.labels_

    all_results = []
    for filename, label in zip(all_differences.keys(), cluster_labels):
        all_results.append({'video': filename, 'cluster': label})

    return all_results

def save_results_to_excel(results, output_file):
    df = pd.DataFrame(results)
    df.to_excel(output_file, index=False)

# 使用范例
folder_path = r'C:\Users\skybl\Desktop\進階專題\New_影片資料夾'
results = process_folder(folder_path)
output_file = 'clustering_DataSet.xlsx'
save_results_to_excel(results, output_file)


程式碼

In [6]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 加載 Excel 文件
data = pd.read_excel('clustering_DataSet.xlsx')

# 確認數據是否正確加載
print(data.head())
print(data.columns)

# 檢查 'video' 和 'activity' 列是否存在
if 'video' in data.columns and 'activity' in data.columns:
    # 按视频名分组，并计算每个视频中不同活动状态的数量
    grouped = data.groupby('video')
    activity_counts = grouped['activity'].value_counts().unstack(fill_value=0)

    # 计算每个视频中活动状态的百分比，并四舍五入到整数位
    activity_percentages = (activity_counts.div(activity_counts.sum(axis=1), axis=0) * 100).round(0).astype(int)

    # 创建 DataFrame 存储 activity_percentages
    activity_percentages_df = pd.DataFrame(activity_percentages)

    # 確保所有列都是數字類型
    activity_percentages_df = activity_percentages_df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    
    # 打印 DataFrame 列名確認
    print(activity_percentages_df.columns)

    # 特徵縮放
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(activity_percentages_df)

    # 應用 K-means 聚類
    kmeans = KMeans(n_clusters=3, random_state=42)
    kmeans.fit(scaled_features)

    # 獲取聚類標籤
    cluster_labels = kmeans.labels_

    # 將聚類結果添加到 DataFrame 中
    activity_percentages_df['cluster'] = cluster_labels

    # 打印結果
    print(activity_percentages_df[['cluster']])

    # 保存結果到 Excel 文件
    activity_percentages_df.to_excel('video_activity_clustering.xlsx', index=True)

else:
    print("列名 'video' 或 'activity' 不存在，請檢查Excel文件中的列名。")


      video activity    difference
0  亂走亂舔.mp4  Passive  2.577000e+13
1  亂走亂舔.mp4  Passive  1.860000e+12
2  亂走亂舔.mp4  Passive  2.561000e+13
3  亂走亂舔.mp4  Passive  2.750000e+12
4  亂走亂舔.mp4  Passive  2.162000e+13
Index(['video', 'activity', 'difference'], dtype='object')
Index(['Aggressive', 'Normal', 'Passive'], dtype='object', name='activity')
activity        cluster
video                  
亂走亂舔.mp4              1
互舔.mp4                2
互舔＋到處晃晃.mp4           2
休息.mp4                2
休息2.mp4               1
休息吃草.mp4              2
休息走路.mp4              2
休息＋人經過轉頭.mp4          1
伸懶腰.mp4               2
動來動去.mp4              2
受水驚嚇.mp4              0
吃草休息.mp4              2
喝水＋聞屁.mp4             1
抬腳抓癢(無裁切版）.mp4        2
搖尾巴＋吃草.mp4            2
正常活動.mp4              2
玩＋聞屁＋舔腳.mp4           2
繞圈走動.mp4              0
舔自己 2.mp4             1
舔自己.mp4               1
走來走去.mp4              0
走來走去＋搖尾巴.mp4          0
走過來給鹿舔.mp4            1
趴下動作.mp4              1


In [7]:
activity_percentages_df[['cluster']]

activity,cluster
video,
亂走亂舔.mp4,1
互舔.mp4,2
互舔＋到處晃晃.mp4,2
休息.mp4,2
休息2.mp4,1
休息吃草.mp4,2
休息走路.mp4,2
休息＋人經過轉頭.mp4,1
伸懶腰.mp4,2


## 包裝成Excel

In [16]:
# 将 new_data DataFrame 保存为 Excel 文件
new_data.to_excel('./new_data.xlsx', index=False)

## 包裝程式

從主要資料庫界定好閥值

In [17]:
import cv2
import numpy as np
import os
import pandas as pd

# 計算活動量%
def calculate_frame_difference(prev_frame, current_frame):
    diff = cv2.absdiff(prev_frame, current_frame)
    return np.sum(diff)

# 開啟鏡頭
def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    if not ret:
        print(f"Cannot read video file {video_path}")
        return []
    
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    differences = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = calculate_frame_difference(prev_frame, current_frame)
        differences.append(diff)
        prev_frame = current_frame

    cap.release()
    return differences

# 定義前、後幾%的資料
def calculate_thresholds(differences):
    high_threshold = np.percentile(differences, 75)
    low_threshold = np.percentile(differences, 25)
    return high_threshold, low_threshold

# 定義分類
def classify_activities(differences, low_threshold, high_threshold):
    activities = []
    for diff in differences:
        if diff >= high_threshold:
            activities.append(('Aggressive', diff))
        elif diff >= low_threshold:
            activities.append(('Normal', diff))
        else:
            activities.append(('Passive', diff))
    return activities

# 讀取全部資料夾的影片
def process_folder(folder_path):
    all_differences = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            all_differences.extend(differences)

    high_threshold, low_threshold = calculate_thresholds(all_differences)
    all_results = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            activities = classify_activities(differences, low_threshold, high_threshold)
            results = [{'video': os.path.basename(video_path), 'activity': act[0], 'difference': act[1]} for act in activities]
            all_results.extend(results)

    return all_results, low_threshold, high_threshold

# 儲存結果
def save_results_to_excel(results, output_file):
    df = pd.DataFrame(results)
    df.to_excel(output_file, index=False)

# 使用範例
folder_path = r'C:\Users\skybl\Desktop\進階專題\New_影片資料夾'
results, low_threshold, high_threshold = process_folder(folder_path)
output_file = 'test1.xlsx'
save_results_to_excel(results, output_file)

# 创建阈值的DataFrame
thresholds_df = pd.DataFrame({'low_threshold': [low_threshold], 'high_threshold': [high_threshold]})

# 打印阈值DataFrame
print(thresholds_df)


   low_threshold  high_threshold
0      707953.75       1950303.0


將影片作判斷

In [18]:
import cv2
import numpy as np
import pandas as pd
import os

def calculate_frame_difference(prev_frame, current_frame):
    diff = cv2.absdiff(prev_frame, current_frame)
    return np.sum(diff)

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    if not ret:
        print(f"Cannot read video file {video_path}")
        return []
    
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    differences = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = calculate_frame_difference(prev_frame, current_frame)
        differences.append(diff)
        prev_frame = current_frame

    cap.release()
    return differences

def classify_differences(differences, low_threshold, high_threshold):
    activities = ['Aggressive' if diff >= high_threshold else 'Normal' if diff >= low_threshold else 'Passive' for diff in differences]
    return activities

def create_dataframe(results, low_threshold, high_threshold):
    video_name = results['video']
    activities = classify_differences(results['differences'], low_threshold, high_threshold)
    flattened_results = [{'video': video_name, 'frame': frame_index + 1, 'difference': diff, 'activity': activity} for frame_index, (diff, activity) in enumerate(zip(results['differences'], activities))]
    df = pd.DataFrame(flattened_results)
    return df

def summarize_activities(df):
    grouped = df.groupby('video')
    activity_counts = grouped['activity'].value_counts().unstack(fill_value=0)
    activity_percentages = (activity_counts.div(activity_counts.sum(axis=1), axis=0) * 100).round(0).astype(int)
    main_activity = activity_percentages.idxmax(axis=1)
    main_activity_percentage = activity_percentages.max(axis=1)
    main_activity[main_activity_percentage <= 75] = 'Normal'
    
    summary_df = pd.DataFrame({
        'video': main_activity.index, 
        'main_activity': main_activity.values,
        'Aggressive': activity_percentages['Aggressive'],
        'Normal': activity_percentages['Normal'],
        'Passive': activity_percentages['Passive']
    })
    
    return summary_df

# Usage example
video_path = r'C:\Users\skybl\Desktop\進階專題\New_影片資料夾\受水驚嚇.mp4'
differences = process_video(video_path)
results = {'video': os.path.basename(video_path), 'differences': differences}

# Assuming thresholds_df is already loaded and contains the thresholds
low_threshold = thresholds_df['low_threshold'][0]
high_threshold = thresholds_df['high_threshold'][0]

df = create_dataframe(results, low_threshold, high_threshold)
summary_df = summarize_activities(df)

print(summary_df)

# Print videos with main activity as Aggressive or Passive
for index, row in summary_df.iterrows():
    if row['main_activity'] in ['Aggressive', 'Passive']:
        print(f"Video: {row['video']}, Main Activity: {row['main_activity']}")


             video main_activity  Aggressive  Normal  Passive
video                                                        
受水驚嚇.mp4  受水驚嚇.mp4    Aggressive          96       4        0
Video: 受水驚嚇.mp4, Main Activity: Aggressive


## 高斯分佈

In [4]:
import cv2
import numpy as np
import os
import pandas as pd
from scipy.stats import norm

def calculate_frame_difference(prev_frame, current_frame):
    diff = cv2.absdiff(prev_frame, current_frame)
    mean_diff = np.mean(diff)  # 使用平均差異
    return mean_diff

def process_video(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    differences = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = calculate_frame_difference(prev_frame, current_frame)
        differences.append(diff)
        prev_frame = current_frame

    cap.release()
    return differences

def calculate_gaussian_thresholds(differences):
    mean = np.mean(differences)
    std_dev = np.std(differences)
    high_threshold = mean + 1.0 * std_dev
    low_threshold = mean - 1.0 * std_dev
    return high_threshold, low_threshold, std_dev

def classify_activities(differences, low_threshold, high_threshold):
    activities = []
    for diff in differences:
        if diff >= high_threshold:
            activities.append(('Aggressive', diff))
        elif diff >= low_threshold:
            activities.append(('Normal', diff))
        else:
            activities.append(('Passive', diff))
    return activities

def process_folder(folder_path):
    all_differences = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            all_differences.extend(differences)

    high_threshold, low_threshold, std_dev = calculate_gaussian_thresholds(all_differences)
    print(f"標準差: {std_dev}")  # 打印標準差
    all_results = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(folder_path, filename)
            differences = process_video(video_path)
            activities = classify_activities(differences, low_threshold, high_threshold)
            results = [{'video': os.path.basename(video_path), 'activity': act[0], 'difference': act[1]} for act in activities]
            all_results.extend(results)

    return all_results

def save_results_to_excel(results, output_file):
    df = pd.DataFrame(results)
    df.to_excel(output_file, index=False)

# 使用范例
folder_path = r'C:\Users\skybl\Desktop\進階專題\New_影片資料夾'
results = process_folder(folder_path)
output_file = 'video_activity_classification.xlsx'
save_results_to_excel(results, output_file)


標準差: 1.645283006935945


In [5]:
import pandas as pd

# 加载Excel文件
data = pd.read_excel('./video_activity_classification.xlsx')

# 确保数据中的列名是正确的，特别是关注 'video' 和 'activity' 列
print(data.head())
print(data.columns)

# 按视频名分组，并计算每个视频中不同活动状态的数量
grouped = data.groupby('video')
activity_counts = grouped['activity'].value_counts().unstack(fill_value=0)

# 创建 DataFrame 存储 activity_counts
activity_counts_df = pd.DataFrame(activity_counts)
activity_counts_df

      video activity  difference
0  亂走亂舔.mp4   Normal    0.636537
1  亂走亂舔.mp4   Normal    0.168668
2  亂走亂舔.mp4   Normal    0.936149
3  亂走亂舔.mp4   Normal    0.262501
4  亂走亂舔.mp4   Normal    0.717984
Index(['video', 'activity', 'difference'], dtype='object')


activity,Aggressive,Normal
video,,
亂走亂舔.mp4,6,347
互舔.mp4,18,506
互舔＋到處晃晃.mp4,63,854
休息.mp4,31,882
休息2.mp4,21,592
休息吃草.mp4,22,451
休息走路.mp4,32,897
休息＋人經過轉頭.mp4,21,591
伸懶腰.mp4,11,290


## 開啟攝像頭，用滑鼠指定某方框區，導入我已寫好的程式做運算

In [ ]:
# import cv2
# import numpy as np
# import os
# import time
# import threading
# import queue
# from tkinter import Tk, Label
# from PIL import Image, ImageTk

# # 全局变量，用于线程间通信
# frame_queue = queue.Queue()
# difference_queue = queue.Queue()

# # 终止标志
# terminate_flag = threading.Event()

# # 摄像头捕获程序
# def capture_videos_from_camera():
#     cap = cv2.VideoCapture(0)  # 打开默认摄像头
#     if not cap.isOpened():
#         print("Error: Could not open video device.")
#         return

#     prev_frame = None

#     while not terminate_flag.is_set():
#         ret, frame = cap.read()
#         if not ret:
#             print("Error: Failed to read frame from camera.")
#             continue

#         # 将帧放入队列
#         if not frame_queue.full():
#             frame_queue.put(frame)

#         # 计算像素差值
#         if prev_frame is not None:
#             gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#             diff = cv2.absdiff(prev_frame, gray)
#             diff_sum = np.sum(diff)

#             # 将差值放入差值队列
#             if not difference_queue.full():
#                 difference_queue.put(diff_sum)

#         prev_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

#     cap.release()
#     print("Capture thread terminated.")

# # 显示摄像头画面的程序
# def display_camera():
#     root = Tk()
#     label = Label(root)
#     label.pack()

#     def update_frame():
#         if not frame_queue.empty():
#             frame = frame_queue.get()
#             frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             img = Image.fromarray(frame_rgb)
#             imgtk = ImageTk.PhotoImage(image=img)
#             label.imgtk = imgtk
#             label.config(image=imgtk)
#         if not terminate_flag.is_set():
#             label.after(10, update_frame)
#         else:
#             root.destroy()

#     update_frame()
#     root.mainloop()
#     print("Display thread terminated.")

# # 状态判断程序
# def classify_differences(differences, low_threshold, high_threshold):
#     avg_diff = np.mean(differences)
#     if avg_diff >= high_threshold:
#         return 'Aggressive'
#     elif avg_diff >= low_threshold:
#         return 'Normal'
#     else:
#         return 'Passive'

# def process_differences(low_threshold, high_threshold):
#     while not terminate_flag.is_set():
#         differences = []
#         start_time = time.time()

#         # 10秒内收集差值数据
#         while time.time() - start_time < 5:
#             if not difference_queue.empty():
#                 diff = difference_queue.get()
#                 differences.append(diff)
#             time.sleep(1)  # 每秒钟收集一次

#         # 2秒计算时间
#         time.sleep(2)
#         if differences:
#             state = classify_differences(differences, low_threshold, high_threshold)
#             print(f"State: {state}")

#     print("Process thread terminated.")

# # 设置保存视频的文件夹路径
# folder_path = r'C:\Users\skybl\Desktop\進階專題\攝像頭保存'

# # 假设阈值已经被加载
# low_threshold = 685304  # 假设这些值已经定义
# high_threshold = 1932026

# # 创建线程
# capture_thread = threading.Thread(target=capture_videos_from_camera)
# process_thread = threading.Thread(target=process_differences, args=(low_threshold, high_threshold))
# display_thread = threading.Thread(target=display_camera)

# # 启动线程
# capture_thread.start()
# process_thread.start()
# display_thread.start()

# # 等待用户中断（例如通过按下Ctrl+C）
# try:
#     while True:
#         time.sleep(1)
# except KeyboardInterrupt:
#     print("Terminating threads...")
#     terminate_flag.set()

# # 等待线程完成
# capture_thread.join()
# process_thread.join()
# display_thread.join()

# print("All threads terminated.")


In [ ]:
# # 假设阈值已经被加载
# low_threshold = 685304  # 假设这些值已经定义
# high_threshold = 1932026

## 評測模型

## 測試